# Fusion Model — Colab / Kaggle

Architecture: **frozen Aniemore backbone** + **BiLSTM on MFCC/mel** + **global audio stats** → MLP → 7 classes

```
Audio
├── Aniemore/wavlm-emotion-russian-resd (frozen) → mean pool → (768,)
├── BiLSTM on MFCC+Δ+ΔΔ+logmel             → mean pool → (512,)
└── Global stats: mean+std of MFCC & mel            → (240,)
         ↓ concat → (1520,)
    Linear → ReLU → Dropout → Linear → 7 classes
```

## 1. GPU check

In [ ]:
import subprocess, sys, os
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available — switch runtime to GPU")

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchaudio', 'torchcodec',
    'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn',
    'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Get the code

**Option A — clone from GitHub:**

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('Working directory:', os.getcwd())

**Option B — Google Drive mount** (Colab only):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/speech_emo_finetune')
# if os.getcwd() not in sys.path:
#     sys.path.insert(0, os.getcwd())

## 4. Config

In [ ]:
CONFIG = 'configs/fusion_wavlm.yaml'
print('Config:', CONFIG)

## 5. Train

In [ ]:
import gc, random, warnings, logging
import torch
import numpy as np
from transformers import AutoFeatureExtractor

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

from src.config import load_config
from src.dataset import get_fusion_dataloaders
from src.models import build_model
from src.trainer import Trainer

for _var in ['train_loader', 'dev_loader', 'test_loader', 'trainer', 'model']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

config = load_config(CONFIG)
set_seed(config.seed)
config.num_workers = 0

# To resume from a checkpoint:
# config.resume_from = 'outputs/fusion_wavlm/checkpoint_epoch_010.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}', flush=True)

processor = AutoFeatureExtractor.from_pretrained(config.processor_name or config.model_name)
train_loader, dev_loader, test_loader = get_fusion_dataloaders(config, processor)
print(f'Train: {len(train_loader.dataset)} | Dev: {len(dev_loader.dataset)} | Test: {len(test_loader.dataset)}', flush=True)

model = build_model(config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)', flush=True)
print(f'  backbone (frozen): {sum(p.numel() for p in model.backbone.parameters()):,}', flush=True)
print(f'  BiLSTM + MLP:      {trainable:,}', flush=True)

_IS_KAGGLE = any(os.environ.get(v) for v in (
    'KAGGLE_KERNEL_RUN_TYPE', 'KAGGLE_URL_BASE', 'KAGGLE_DATA_PROXY_TOKEN'
))

def _colab_download(path):
    if _IS_KAGGLE:
        print(f'Checkpoint saved: {path.name} (download via Kaggle Output tab)', flush=True)
        return
    try:
        from google.colab import files
        print(f'Downloading {path.name}...', flush=True)
        files.download(str(path))
    except BaseException:
        print(f'Checkpoint saved: {path}', flush=True)

trainer = Trainer(model, config, train_loader, dev_loader, test_loader, device)
test_metrics = trainer.fit(checkpoint_callback=_colab_download)

## 5b. Download checkpoints

- **Colab**: run this cell to download all saved checkpoints
- **Kaggle**: download from the Output tab

In [ ]:
import pathlib
try:
    from google.colab import files
    output_dir = pathlib.Path(config.output_dir)
    for ckpt in sorted(output_dir.glob('checkpoint_epoch_*.pt')):
        print(f'Downloading {ckpt.name}...')
        files.download(str(ckpt))
except ImportError:
    print('Not in Colab — checkpoints are in:', config.output_dir)

## 6. Training curves

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import yaml

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
metrics_path = pathlib.Path(cfg.get('output_dir', 'outputs/fusion_wavlm')) / 'metrics.jsonl'

records    = [json.loads(l) for l in open(metrics_path) if 'epoch' in json.loads(l)]
epochs     = [r['epoch'] for r in records]
train_loss = [r['train_loss'] for r in records]
dev_f1     = [r['dev_f1_weighted'] for r in records]
dev_acc    = [r['dev_accuracy'] for r in records]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, marker='o')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, dev_f1, marker='o', label='F1 weighted')
axes[1].plot(epochs, dev_acc, marker='s', linestyle='--', label='Accuracy')
axes[1].set_title('Dev Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(cfg.get('run_name', 'fusion_wavlm'))
plt.tight_layout()
plt.show()

## 7. Test results + confusion matrix

In [ ]:
import numpy as np
import seaborn as sns

all_records = [json.loads(l) for l in open(metrics_path)]
test_rec = [r for r in all_records if r.get('split') == 'test'][-1]

print(f"accuracy          {test_rec['accuracy']:.4f}")
print(f"weighted_accuracy {test_rec['weighted_accuracy']:.4f}")
print(f"f1_macro          {test_rec['f1_macro']:.4f}")
print(f"f1_weighted       {test_rec['f1_weighted']:.4f}")
print()
print(f"{'Class':12s} {'P':>6} {'R':>6} {'F1':>6} {'n':>5}")
for cls, vals in test_rec['per_class'].items():
    print(f"{cls:12s} {vals['precision']:6.3f} {vals['recall']:6.3f} {vals['f1']:6.3f} {int(vals['support']):5d}")

cm = np.array(test_rec['confusion_matrix'])
label_names = list(test_rec['per_class'].keys())

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {cfg.get("run_name", "")}')
plt.tight_layout()
plt.show()